[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/16.%20The%20Storage%20Location%20Assignment%20Problem%20%28SLAP%29/P16-Tier-6_executed.ipynb)

---


# 16. The Storage Location Assignment Problem (SLAP)
## Tier 6: Distributed Intelligence and Autonomous Agents

This notebook models storage assignment as a sequence of auctions between product agents and storage-location agents. Each product agent evaluates bids that satisfy its cost and service requirements, then accepts the highest-scoring bid. The winning location records the resulting contract and updates its available capacity. The simulation also tracks agent reputation, assignment activity, and selected patterns in the resulting assignments.

### Goals, inputs, and outputs

The model is designed to demonstrate how local agent decisions and an auction market can coordinate assignments without computing a centralized global optimum.

- **Inputs:** location and product populations, location capabilities and coordinates, product categories and demand velocities, and a demand scenario.
- **Process:** generate bids, screen and score them from the product's perspective, form a contract with the selected location, and record simulation metrics.
- **Outputs:** assignments and contracts, market statistics, performance indicators, detected assignment patterns, and plots summarizing the simulation.

### Scope and modeling assumptions

This is an illustrative agent-based simulation, not a validated operational SLAP solver. The auction process is the decision method studied here; its results are not guaranteed to be globally optimal or to be a game-theoretic equilibrium. The simulation samples a limited number of locations for each auction rather than comparing every possible product-location pair.

Each accepted product contract consumes one unit of location capacity. The generated product size is retained as an attribute but does not affect capacity consumption. Location coordinates are used to calculate proximity to the origin, which represents an input/output point. Assignment is made once for each product during a run: the model does not relocate already assigned products, model inventory replenishment, or enforce detailed warehouse geometry. Reputation updates use simulated contract outcomes, and the demand scenarios change product velocity without modeling a time-varying order stream.

The nominal population sizes and the performance claims sometimes associated with large distributed systems are not evidence of performance in this model. No centralized benchmark, computational-complexity comparison, or measured adaptation-speed comparison is computed here.

### Auction decision rule

For each auction, a location agent submits a bid based on its cost, current load, product velocity, and specialization. A product agent discards bids that exceed its maximum acceptable cost or fall below its minimum reliability. Among the remaining bids, it selects the one with the largest product utility score.

Let \(B_p\) be the bids received for product \(p\), and let \(F_p\subseteq B_p\) be the bids that satisfy the product's cost and reliability requirements. The selected bid is

$$
b_p^* = \underset{b\in F_p}{\operatorname{argmax}}\; U_p(b),
$$

where the product utility score is

$$
U_p(b) = 0.4\left(1-\frac{c_b}{m_p}\right)
       + 0.3\frac{s_b}{10}
       + 0.2r_b
       + 0.1q_{p,b}.
$$

Here, \(c_b\) is the bid price, \(m_p\) is the product's maximum acceptable cost, \(s_b\) is the offered processing speed, \(r_b\) is the offered reliability, and \(q_{p,b}\) is a specialization bonus: it is \(0.2\) when the location specializes in the product category and \(0\) otherwise. If \(F_p\) is empty, no contract is formed for that product in that auction. The code implements this rule in `ProductAgent.evaluate_bids`.

The location also calculates a separate weighted utility from its own perspective. That score is recorded on its bid, but the auction winner is selected by the product agent's score above. This distinction reflects the two sides of the market; a location's bid score is not the auction's selection objective.

For \(K\) auctions with at most \(B\) bids evaluated per auction, bid evaluation takes \(O(KB)\) time, in addition to the work of generating bids and maintaining assignments. This is not a complexity comparison with a centralized optimization model. Sampling limits the number of bids considered, so assignments depend in part on which locations are sampled.


In [ ]:
from collections import defaultdict, deque
from dataclasses import dataclass
from datetime import datetime, timedelta
from enum import Enum
import time
from typing import Any, Deque, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
class AgentState(Enum):
    """Operational states used by location agents."""

    ACTIVE = "active"
    NEGOTIATING = "negotiating"
    CONTRACTED = "contracted"
    IDLE = "idle"
    MAINTENANCE = "maintenance"


@dataclass
class AgentCapability:
    """Capacity and service characteristics offered by a location."""

    capacity: float
    speed: float
    reliability: float
    cost_per_unit: float
    specialization: List[str]


@dataclass
class UtilityFunction:
    """Weights used by a location to score a potential assignment."""

    weight_cost: float = 0.3
    weight_speed: float = 0.25
    weight_reliability: float = 0.2
    weight_proximity: float = 0.15
    weight_reputation: float = 0.1


@dataclass
class Contract:
    """A bilateral agreement formed after a successful auction."""

    contract_id: str
    buyer_id: str
    seller_id: str
    product_id: str
    terms: Dict[str, Any]
    duration: timedelta
    penalty_rate: float
    created_at: datetime
    status: str = "active"


@dataclass
class Bid:
    """A location agent's offer in a product auction."""

    bid_id: str
    bidder_id: str
    product_id: str
    price: float
    utility_score: float
    terms: Dict[str, Any]
    timestamp: datetime

In [ ]:
class LocationAgent:
    """Storage-location agent that offers capacity to product agents."""

    def __init__(
        self,
        agent_id: str,
        location_type: str,
        position: Tuple[float, float],
        capability: AgentCapability,
        utility_function: UtilityFunction,
    ) -> None:
        if capability.capacity <= 0:
            raise ValueError("Location capacity must be positive.")

        self.agent_id = agent_id
        self.location_type = location_type
        self.position = position
        self.capability = capability
        self.utility_function = utility_function
        self.state = AgentState.ACTIVE
        self.current_load = 0.0
        self.contracts: List[Contract] = []
        self.reputation_score = 0.5
        self.available_capacity = capability.capacity
        self.asking_price = capability.cost_per_unit
        self.success_history: Deque[bool] = deque(maxlen=100)
        self.profit_history: Deque[float] = deque(maxlen=100)

    def calculate_utility(self, product_agent: "ProductAgent", bid_price: float) -> float:
        """Score an offer from the location agent's perspective."""
        cost_utility = 1.0 - bid_price / (self.capability.cost_per_unit * 2.0)
        cost_utility = max(0.0, min(1.0, cost_utility))

        speed_utility = max(0.0, min(1.0, self.capability.speed / 10.0))
        reliability_utility = max(0.0, min(1.0, self.capability.reliability))

        distance_to_io = float(np.hypot(self.position[0], self.position[1]))
        proximity_utility = max(0.0, min(1.0, 1.0 - distance_to_io / 1000.0))
        reputation_utility = self.reputation_score

        return (
            self.utility_function.weight_cost * cost_utility
            + self.utility_function.weight_speed * speed_utility
            + self.utility_function.weight_reliability * reliability_utility
            + self.utility_function.weight_proximity * proximity_utility
            + self.utility_function.weight_reputation * reputation_utility
        )

    def submit_bid(self, product_agent: "ProductAgent", auction_id: str) -> Bid:
        """Create a price offer adjusted for load and product characteristics."""
        load_factor = self.current_load / self.capability.capacity
        demand_multiplier = 1.0 + load_factor * 0.5

        if product_agent.is_high_velocity:
            demand_multiplier *= 1.2

        if product_agent.product_category in self.capability.specialization:
            demand_multiplier *= 0.9

        bid_price = self.capability.cost_per_unit * demand_multiplier
        utility_score = self.calculate_utility(product_agent, bid_price)

        return Bid(
            bid_id=f"{self.agent_id}_{auction_id}_{int(time.time() * 1_000_000)}",
            bidder_id=self.agent_id,
            product_id=product_agent.product_id,
            price=bid_price,
            utility_score=utility_score,
            terms={
                "service_level": self.capability.reliability,
                "processing_speed": self.capability.speed,
                "specialization": self.capability.specialization,
            },
            timestamp=datetime.now(),
        )

    def execute_contract(self, contract: Contract) -> None:
        """Record a contract and update capacity and operational state."""
        if self.available_capacity < 1.0:
            raise ValueError("The location does not have capacity for another product.")

        self.contracts.append(contract)
        self.current_load += 1.0
        self.available_capacity = max(0.0, self.capability.capacity - self.current_load)
        self.state = (
            AgentState.ACTIVE
            if self.available_capacity > 0.0
            else AgentState.CONTRACTED
        )

    def update_reputation(self, success: bool, profit: float) -> None:
        """Move reputation toward the observed success rate after ten outcomes."""
        self.success_history.append(success)
        self.profit_history.append(profit)

        if len(self.success_history) >= 10:
            recent_success_rate = sum(list(self.success_history)[-10:]) / 10.0
            self.reputation_score = (
                0.7 * self.reputation_score + 0.3 * recent_success_rate
            )
            self.reputation_score = max(0.0, min(1.0, self.reputation_score))


class ProductAgent:
    """Product agent that screens bids and selects a location."""

    def __init__(
        self,
        product_id: str,
        velocity: float,
        size: float,
        category: str,
        is_high_velocity: bool = False,
    ) -> None:
        self.product_id = product_id
        self.velocity = velocity
        self.size = size
        self.product_category = category
        self.is_high_velocity = is_high_velocity
        self.current_location: Optional[str] = None
        self.contracts: List[Contract] = []
        self.utility_function = UtilityFunction()
        self.min_reliability = 0.8
        self.max_cost = velocity * 10.0
        self.preferred_specialization = [category]

    def evaluate_bids(self, bids: List[Bid]) -> Tuple[Optional[Bid], float]:
        """Filter infeasible offers and return the feasible offer with highest score."""
        if not bids:
            return None, 0.0

        best_bid: Optional[Bid] = None
        best_score = -float("inf")

        for bid in bids:
            if bid.price > self.max_cost:
                continue
            if bid.terms["service_level"] < self.min_reliability:
                continue

            cost_score = 1.0 - bid.price / self.max_cost
            speed_score = bid.terms["processing_speed"] / 10.0
            reliability_score = bid.terms["service_level"]
            specialization_bonus = (
                0.2
                if self.product_category in bid.terms["specialization"]
                else 0.0
            )

            total_score = (
                0.4 * cost_score
                + 0.3 * speed_score
                + 0.2 * reliability_score
                + 0.1 * specialization_bonus
            )

            if total_score > best_score:
                best_score = total_score
                best_bid = bid

        if best_bid is None:
            return None, 0.0
        return best_bid, best_score

    def accept_contract(
        self, bid: Bid, location_agent: LocationAgent
    ) -> Contract:
        """Create a contract and register it with both participating agents."""
        contract = Contract(
            contract_id=f"CONTRACT_{self.product_id}_{location_agent.agent_id}",
            buyer_id=self.product_id,
            seller_id=location_agent.agent_id,
            product_id=self.product_id,
            terms={
                "price": bid.price,
                "service_level": bid.terms["service_level"],
                "processing_speed": bid.terms["processing_speed"],
            },
            duration=timedelta(days=30),
            penalty_rate=0.1,
            created_at=datetime.now(),
        )

        location_agent.execute_contract(contract)
        self.contracts.append(contract)
        self.current_location = location_agent.agent_id
        return contract

## Market mechanism

The market creates one auction for each product selected for assignment. Location agents submit offers, and the product agent chooses among feasible bids. The market records the winning offer and basic price-discovery statistics; it does not centrally optimize assignments across products.


In [ ]:
class DistributedMarket:
    """Auction market used to coordinate product and location agents."""

    def __init__(self, market_id: str) -> None:
        self.market_id = market_id
        self.active_auctions: Dict[str, Dict[str, Any]] = {}
        self.market_history: List[Dict[str, Any]] = []
        self.price_discovery: Dict[str, List[float]] = defaultdict(list)

    def create_auction(
        self, product_agent: ProductAgent, duration_minutes: int = 5
    ) -> str:
        """Open an auction and return its identifier."""
        auction_id = f"AUCTION_{product_agent.product_id}"
        start_time = datetime.now()
        self.active_auctions[auction_id] = {
            "auction_id": auction_id,
            "product_agent": product_agent,
            "bids": [],
            "start_time": start_time,
            "end_time": start_time + timedelta(minutes=duration_minutes),
            "status": "active",
            "winner": None,
        }
        return auction_id

    def submit_bid(self, auction_id: str, bid: Bid) -> bool:
        """Add a bid only while its auction is active and within its time limit."""
        auction = self.active_auctions.get(auction_id)
        if auction is None:
            return False

        if auction["status"] != "active" or datetime.now() >= auction["end_time"]:
            return False

        auction["bids"].append(bid)
        return True

    def close_auction(self, auction_id: str) -> Optional[Bid]:
        """Close an auction, select the product's preferred bid, and record it."""
        auction = self.active_auctions.pop(auction_id, None)
        if auction is None:
            return None

        auction["status"] = "closed"
        product_agent = auction["product_agent"]
        best_bid, _ = product_agent.evaluate_bids(auction["bids"])

        if best_bid is None:
            return None

        auction["winner"] = best_bid.bidder_id
        self.price_discovery[product_agent.product_category].append(best_bid.price)
        self.market_history.append(
            {
                "auction_id": auction_id,
                "product_id": product_agent.product_id,
                "category": product_agent.product_category,
                "winning_bid": best_bid.price,
                "num_bids": len(auction["bids"]),
                "timestamp": datetime.now(),
            }
        )
        return best_bid

    def get_market_stats(self) -> Dict[str, Any]:
        """Summarize recorded auctions and category-level winning prices."""
        if not self.market_history:
            return {"status": "no_data"}

        total_auctions = len(self.market_history)
        average_bids = float(
            np.mean([entry["num_bids"] for entry in self.market_history])
        )
        price_trends: Dict[str, Dict[str, Any]] = {}

        for category, prices in self.price_discovery.items():
            if prices:
                price_trends[category] = {
                    "current_price": prices[-1],
                    "avg_price": float(np.mean(prices)),
                    "price_volatility": float(np.std(prices)),
                    "trend": (
                        "up"
                        if len(prices) >= 2 and prices[-1] > prices[-2]
                        else "down"
                        if len(prices) >= 2 and prices[-1] < prices[-2]
                        else "unchanged"
                    ),
                }

        return {
            "total_auctions": total_auctions,
            "active_auctions": len(self.active_auctions),
            "avg_bids_per_auction": average_bids,
            "price_trends": price_trends,
            "market_efficiency": min(1.0, average_bids / 5.0),
        }

## Ecosystem initialization

The ecosystem creates the location and product populations and assigns each location a capability profile. Location types differ in capacity, speed, reliability, cost, and specialization. Product velocity is sampled from the specified ranges, with a small high-velocity group and a larger low-velocity group. These generated attributes feed directly into bid prices, feasibility checks, and utility scores.

The default population sizes preserve the example scale. The simulation limits each auction cycle to at most 1,000 products and each product auction to at most 50 sampled locations. Random sampling is part of this illustrative market process; no random seed is specified, so independently run simulations need not produce identical assignments.


In [ ]:
class AutonomousEcosystem:
    """Agent-based simulation of auction-driven storage assignment."""

    def __init__(self, n_locations: int = 500, n_products: int = 100000) -> None:
        if n_locations <= 0 or n_products <= 0:
            raise ValueError("The ecosystem must contain locations and products.")

        self.n_locations = n_locations
        self.n_products = n_products
        self.location_agents: List[LocationAgent] = []
        self.product_agents: List[ProductAgent] = []
        self.market = DistributedMarket("SLAP_MARKET_001")
        self.current_time = datetime.now()
        self.simulation_running = False
        self.performance_history: List[Dict[str, Any]] = []
        self.emergent_behaviors: List[Dict[str, Any]] = []
        self._processed_contract_ids: set = set()
        self._initialize_ecosystem()

    def _initialize_ecosystem(self) -> None:
        """Generate agent populations and their attributes."""
        print("Initializing the autonomous ecosystem:")
        print(f"  Location agents: {self.n_locations:,}")
        print(f"  Product agents: {self.n_products:,}")

        location_types = ["premium", "standard", "bulk"]
        type_distribution = [0.1, 0.3, 0.6]

        for index in range(self.n_locations):
            location_type = str(
                np.random.choice(location_types, p=type_distribution)
            )
            position = (
                float(np.random.uniform(0, 1000)),
                float(np.random.uniform(0, 500)),
            )

            if location_type == "premium":
                capability = AgentCapability(
                    capacity=100.0,
                    speed=9.0,
                    reliability=0.98,
                    cost_per_unit=15.0,
                    specialization=["electronics", "pharma", "luxury"],
                )
            elif location_type == "standard":
                capability = AgentCapability(
                    capacity=200.0,
                    speed=6.0,
                    reliability=0.92,
                    cost_per_unit=8.0,
                    specialization=["clothing", "books", "home"],
                )
            else:
                capability = AgentCapability(
                    capacity=500.0,
                    speed=3.0,
                    reliability=0.85,
                    cost_per_unit=3.0,
                    specialization=["furniture", "sports", "seasonal"],
                )

            utility_function = UtilityFunction(
                weight_cost=float(np.random.uniform(0.2, 0.4)),
                weight_speed=float(np.random.uniform(0.2, 0.3)),
                weight_reliability=float(np.random.uniform(0.15, 0.25)),
                weight_proximity=float(np.random.uniform(0.1, 0.2)),
                weight_reputation=float(np.random.uniform(0.05, 0.15)),
            )
            self.location_agents.append(
                LocationAgent(
                    agent_id=f"LOC_{index:04d}",
                    location_type=location_type,
                    position=position,
                    capability=capability,
                    utility_function=utility_function,
                )
            )

        product_categories = [
            "electronics",
            "pharma",
            "clothing",
            "books",
            "home",
            "furniture",
            "sports",
            "seasonal",
            "luxury",
            "toys",
        ]

        for index in range(self.n_products):
            category = str(np.random.choice(product_categories))
            draw = np.random.random()

            if draw < 0.05:
                velocity = float(np.random.uniform(80, 100))
                is_high_velocity = True
            elif draw < 0.20:
                velocity = float(np.random.uniform(40, 80))
                is_high_velocity = False
            else:
                velocity = float(np.random.uniform(5, 40))
                is_high_velocity = False

            self.product_agents.append(
                ProductAgent(
                    product_id=f"SKU_{index:06d}",
                    velocity=velocity,
                    size=float(np.random.uniform(0.5, 5.0)),
                    category=category,
                    is_high_velocity=is_high_velocity,
                )
            )

        high_velocity_count = sum(
            product.is_high_velocity for product in self.product_agents
        )
        premium_count = sum(
            location.location_type == "premium"
            for location in self.location_agents
        )
        print("Ecosystem initialized.")
        print(f"  High-velocity products: {high_velocity_count:,}")
        print(f"  Premium locations: {premium_count:,}")

    def run_simulation(
        self, duration_minutes: int = 30, demand_scenario: str = "normal"
    ) -> None:
        """Run successive auction cycles over the requested simulated duration."""
        if duration_minutes < 0:
            raise ValueError("Simulation duration cannot be negative.")

        print("\nStarting the autonomous ecosystem simulation")
        print(f"Duration: {duration_minutes} minutes")
        print(f"Demand scenario: {demand_scenario}")

        self.simulation_running = True
        start_time = self.current_time
        end_time = start_time + timedelta(minutes=duration_minutes)
        self._apply_demand_scenario(demand_scenario)
        auction_cycle = 0

        while self.current_time < end_time and self.simulation_running:
            contracts_formed = self._run_auction_cycle()
            auction_cycle += 1
            self._update_agent_learning()

            emergent_behavior = self._detect_emergent_behaviors()
            if emergent_behavior is not None:
                self.emergent_behaviors.append(emergent_behavior)

            performance = self._calculate_performance()
            self.performance_history.append(
                {
                    "timestamp": self.current_time,
                    "auction_cycle": auction_cycle,
                    "contracts_formed": contracts_formed,
                    "performance": performance,
                }
            )

            if auction_cycle % 10 == 0:
                elapsed = (self.current_time - start_time).total_seconds() / 60.0
                print(
                    f"[{self.current_time.strftime('%H:%M:%S')}] "
                    f"Cycle {auction_cycle}: {elapsed:.0f}/{duration_minutes} min"
                )
                print(f"  Contracts formed: {contracts_formed:,}")
                print(f"  Assignment rate: {performance['assignment_rate']:.3f}")
                if emergent_behavior is not None:
                    print(f"  Detected pattern: {emergent_behavior['type']}")

            self.current_time += timedelta(seconds=30)

        self.simulation_running = False
        print(f"\nSimulation completed. Total cycles: {auction_cycle}")

    def _apply_demand_scenario(self, scenario: str) -> None:
        """Adjust product demand attributes for a named scenario."""
        if scenario == "black_friday":
            print("Applying the Black Friday demand scenario.")
            for product in self.product_agents:
                if product.product_category in ["electronics", "luxury"]:
                    product.velocity *= 3.5
                    product.is_high_velocity = True
                    product.max_cost = product.velocity * 12.0
                elif product.product_category in ["clothing", "sports"]:
                    product.velocity *= 2.0
        elif scenario == "flu_season":
            print("Applying the flu-season demand scenario.")
            for product in self.product_agents:
                if product.product_category in ["pharma", "healthcare"]:
                    product.velocity *= 4.0
                    product.is_high_velocity = True
        elif scenario != "normal":
            raise ValueError(
                "Demand scenario must be 'normal', 'black_friday', or 'flu_season'."
            )

    def _run_auction_cycle(self) -> int:
        """Auction a sample of currently unassigned products."""
        products_needing_contracts = [
            product
            for product in self.product_agents
            if product.current_location is None
        ]
        auction_count = min(1000, len(products_needing_contracts))
        if auction_count == 0:
            return 0

        selected_indices = np.random.choice(
            len(products_needing_contracts), auction_count, replace=False
        )
        selected_products = [
            products_needing_contracts[int(index)] for index in selected_indices
        ]

        contracts_formed = 0
        for product in selected_products:
            available_locations = [
                location
                for location in self.location_agents
                if location.available_capacity >= 1.0
                and location.state == AgentState.ACTIVE
            ]
            if not available_locations:
                break

            auction_id = self.market.create_auction(product, duration_minutes=1)
            bidder_count = min(50, len(available_locations))
            bidder_indices = np.random.choice(
                len(available_locations), bidder_count, replace=False
            )

            for index in bidder_indices:
                location = available_locations[int(index)]
                bid = location.submit_bid(product, auction_id)
                self.market.submit_bid(auction_id, bid)

            winning_bid = self.market.close_auction(auction_id)
            if winning_bid is None:
                continue

            winning_location = next(
                location
                for location in self.location_agents
                if location.agent_id == winning_bid.bidder_id
            )
            product.accept_contract(winning_bid, winning_location)
            contracts_formed += 1

        return contracts_formed

    def _update_agent_learning(self) -> None:
        """Simulate outcomes once for each newly formed location contract."""
        for location in self.location_agents:
            for contract in location.contracts:
                if contract.contract_id in self._processed_contract_ids:
                    continue

                success = bool(
                    np.random.random() < location.capability.reliability
                )
                profit = contract.terms["price"] * float(
                    np.random.uniform(0.8, 1.2)
                )
                location.update_reputation(success, profit)
                self._processed_contract_ids.add(contract.contract_id)

    def _detect_emergent_behaviors(self) -> Optional[Dict[str, Any]]:
        """Identify simple concentration patterns in current assignments."""
        location_assignments: Dict[str, List[ProductAgent]] = defaultdict(list)
        for product in self.product_agents:
            if product.current_location is not None:
                location_assignments[product.current_location].append(product)

        high_velocity_locations = [
            location_id
            for location_id, products in location_assignments.items()
            if products
            and sum(product.is_high_velocity for product in products) / len(products)
            > 0.7
        ]
        if len(high_velocity_locations) > len(self.location_agents) * 0.1:
            return {
                "type": "high_velocity_concentration",
                "description": (
                    "High-velocity products are concentrated in "
                    f"{len(high_velocity_locations)} locations."
                ),
                "timestamp": self.current_time,
                "affected_locations": len(high_velocity_locations),
            }

        category_clusters: Dict[str, List[str]] = defaultdict(list)
        for location_id, products in location_assignments.items():
            if len(products) < 5:
                continue
            categories = [product.product_category for product in products]
            dominant_category = max(set(categories), key=categories.count)
            if categories.count(dominant_category) / len(categories) > 0.6:
                category_clusters[dominant_category].append(location_id)

        if len(category_clusters) > 3:
            return {
                "type": "category_concentration",
                "description": (
                    "Assignments show concentration across "
                    f"{len(category_clusters)} product categories."
                ),
                "timestamp": self.current_time,
                "clusters": dict(category_clusters),
            }

        return None

    def _calculate_performance(self) -> Dict[str, float]:
        """Compute assignment, load-dispersion, and market activity indicators."""
        assigned_products = sum(
            product.current_location is not None for product in self.product_agents
        )
        assignment_rate = assigned_products / len(self.product_agents)

        location_loads = [
            location.current_load / location.capability.capacity
            for location in self.location_agents
        ]
        load_balance = 1.0 - float(np.std(location_loads))

        market_stats = self.market.get_market_stats()
        market_efficiency = float(market_stats.get("market_efficiency", 0.0))
        efficiency = (
            0.4 * assignment_rate + 0.3 * load_balance + 0.3 * market_efficiency
        )

        return {
            "assignment_rate": assignment_rate,
            "load_balance": load_balance,
            "market_efficiency": market_efficiency,
            "efficiency": efficiency,
        }

    def get_ecosystem_summary(self) -> Dict[str, Any]:
        """Return a summary of the simulation state and recorded outcomes."""
        if not self.performance_history:
            return {"status": "no_simulation_data"}

        final_performance = self.performance_history[-1]["performance"]
        initial_performance = self.performance_history[0]["performance"]
        efficiency_improvement = (
            final_performance["efficiency"] - initial_performance["efficiency"]
        )
        active_locations = sum(
            location.state == AgentState.ACTIVE for location in self.location_agents
        )
        average_reputation = float(
            np.mean([location.reputation_score for location in self.location_agents])
        )
        all_contracts = [
            contract
            for product in self.product_agents
            for contract in product.contracts
        ]
        average_contract_value = (
            float(np.mean([contract.terms["price"] for contract in all_contracts]))
            if all_contracts
            else 0.0
        )
        behavior_counts: Dict[str, int] = defaultdict(int)
        for behavior in self.emergent_behaviors:
            behavior_counts[behavior["type"]] += 1

        return {
            "simulation_cycles": len(self.performance_history),
            "total_contracts": len(all_contracts),
            "avg_contract_value": average_contract_value,
            "final_efficiency": final_performance["efficiency"],
            "efficiency_improvement": efficiency_improvement,
            "assignment_rate": final_performance["assignment_rate"],
            "load_balance": final_performance["load_balance"],
            "market_efficiency": final_performance["market_efficiency"],
            "active_locations": active_locations,
            "avg_reputation": average_reputation,
            "emergent_behaviors": dict(behavior_counts),
            "market_stats": self.market.get_market_stats(),
        }

## Run the Black Friday scenario

The simulation advances in 30-second increments of simulated time. Each cycle considers up to 1,000 unassigned products; for each product, up to 50 currently available locations are sampled to submit bids. A product receives a contract only if at least one sampled bid passes its feasibility checks. The location remains available while it has at least one unit of capacity remaining.

The market and agent counts below match the example configuration. They define the simulation input; they do not imply a particular outcome or establish a performance guarantee.


In [ ]:
print("Creating the autonomous self-optimizing ecosystem.")
ecosystem = AutonomousEcosystem(
    n_locations=500,
    n_products=100000,
)
ecosystem.run_simulation(duration_minutes=30, demand_scenario="black_friday")

## Visualize and summarize results

The panels report measurements from this simulation run: the composite indicator over auction cycles, contracts formed per cycle, location reputation, a location-type score, detected pattern events, and market prices or summary statistics. The composite indicator is a descriptive weighted metric defined in `_calculate_performance`; it is not a percentage of a centralized optimum. The location-type score is also descriptive and combines mean reputation with unused capacity.


In [ ]:
def visualize_ecosystem_results(ecosystem: AutonomousEcosystem) -> None:
    """Plot simulation indicators and print a descriptive run summary."""
    summary = ecosystem.get_ecosystem_summary()
    if summary.get("status") == "no_simulation_data":
        print("No simulation data is available to visualize.")
        return

    figure, axes = plt.subplots(2, 3, figsize=(18, 12))
    figure.suptitle(
        "Autonomous Ecosystem: Simulation Results",
        fontsize=16,
        fontweight="bold",
    )

    cycles = [entry["auction_cycle"] for entry in ecosystem.performance_history]
    efficiencies = [
        entry["performance"]["efficiency"]
        for entry in ecosystem.performance_history
    ]
    axes[0, 0].plot(cycles, efficiencies, color="blue", marker="o", markersize=3)
    axes[0, 0].fill_between(cycles, efficiencies, alpha=0.2, color="blue")
    axes[0, 0].set_title("Composite Indicator by Cycle")
    axes[0, 0].set_xlabel("Auction cycle")
    axes[0, 0].set_ylabel("Indicator value")
    axes[0, 0].set_ylim(0, 1)
    axes[0, 0].grid(True, alpha=0.3)

    contracts_per_cycle = [
        entry["contracts_formed"] for entry in ecosystem.performance_history
    ]
    axes[0, 1].plot(cycles, contracts_per_cycle, color="green", alpha=0.8)
    axes[0, 1].fill_between(cycles, contracts_per_cycle, alpha=0.2, color="green")
    axes[0, 1].set_title("Contracts Formed per Cycle")
    axes[0, 1].set_xlabel("Auction cycle")
    axes[0, 1].set_ylabel("Contracts")
    axes[0, 1].grid(True, alpha=0.3)

    reputations = [
        location.reputation_score for location in ecosystem.location_agents
    ]
    axes[0, 2].hist(
        reputations, bins=20, color="skyblue", edgecolor="black", alpha=0.8
    )
    axes[0, 2].set_title("Location Reputation Distribution")
    axes[0, 2].set_xlabel("Reputation score")
    axes[0, 2].set_ylabel("Number of locations")
    axes[0, 2].grid(True, alpha=0.3)
    axes[0, 2].axvline(
        np.mean(reputations),
        color="red",
        linestyle="--",
        label=f"Mean: {np.mean(reputations):.3f}",
    )
    axes[0, 2].legend()

    location_types = ["premium", "standard", "bulk"]
    type_scores = []
    for location_type in location_types:
        agents = [
            location
            for location in ecosystem.location_agents
            if location.location_type == location_type
        ]
        if not agents:
            type_scores.append(0.0)
            continue
        mean_reputation = float(
            np.mean([location.reputation_score for location in agents])
        )
        mean_load = float(
            np.mean(
                [
                    location.current_load / location.capability.capacity
                    for location in agents
                ]
            )
        )
        type_scores.append(mean_reputation * (1.0 - mean_load))

    bars = axes[1, 0].bar(
        location_types,
        type_scores,
        color=["gold", "lightblue", "lightgreen"],
        alpha=0.8,
    )
    axes[1, 0].set_title("Descriptive Score by Location Type")
    axes[1, 0].set_xlabel("Location type")
    axes[1, 0].set_ylabel("Mean reputation × unused-capacity fraction")
    axes[1, 0].grid(True, axis="y", alpha=0.3)
    label_offset = max(type_scores, default=0.0) * 0.01
    for bar, score in zip(bars, type_scores):
        axes[1, 0].text(
            bar.get_x() + bar.get_width() / 2.0,
            bar.get_height() + label_offset,
            f"{score:.3f}",
            ha="center",
            va="bottom",
        )

    behavior_axis = axes[1, 1]
    if ecosystem.emergent_behaviors:
        behavior_types = sorted(
            {behavior["type"] for behavior in ecosystem.emergent_behaviors}
        )
        colors = plt.cm.Set3(np.linspace(0, 1, len(behavior_types)))
        behavior_colors = dict(zip(behavior_types, colors))
        for index, behavior in enumerate(ecosystem.emergent_behaviors):
            behavior_axis.scatter(
                behavior["timestamp"],
                index,
                color=behavior_colors[behavior["type"]],
                label=behavior["type"],
            )
        handles, labels = behavior_axis.get_legend_handles_labels()
        unique_handles = dict(zip(labels, handles))
        behavior_axis.legend(unique_handles.values(), unique_handles.keys())
        behavior_axis.set_xlabel("Simulation timestamp")
        behavior_axis.set_ylabel("Detected event index")
    else:
        behavior_axis.text(
            0.5,
            0.5,
            "No concentration patterns detected",
            ha="center",
            va="center",
            transform=behavior_axis.transAxes,
        )
    behavior_axis.set_title("Detected Assignment Patterns")
    behavior_axis.grid(True, alpha=0.3)

    market_axis = axes[1, 2]
    market_stats = summary["market_stats"]
    price_trends = market_stats.get("price_trends", {})
    if price_trends:
        categories = list(price_trends)
        current_prices = [
            price_trends[category]["current_price"] for category in categories
        ]
        market_axis.bar(categories, current_prices, color="orange", alpha=0.8)
        market_axis.set_title("Latest Winning Price by Category")
        market_axis.set_ylabel("Price (model units)")
        market_axis.tick_params(axis="x", rotation=45)
    else:
        labels = ["Auctions", "Active", "Market indicator"]
        values = [
            market_stats.get("total_auctions", 0),
            market_stats.get("active_auctions", 0),
            market_stats.get("market_efficiency", 0.0),
        ]
        market_axis.bar(labels, values, color="purple", alpha=0.8)
        market_axis.set_title("Market Summary")
        market_axis.set_ylabel("Value")
        market_axis.tick_params(axis="x", rotation=30)
    market_axis.grid(True, axis="y", alpha=0.3)

    figure.tight_layout()
    plt.show()

    print("\nSimulation summary")
    print("=" * 60)
    print(
        f"Ecosystem: {ecosystem.n_locations:,} location agents, "
        f"{ecosystem.n_products:,} product agents"
    )
    print(f"Simulation cycles: {summary['simulation_cycles']}")
    print(f"Contracts formed: {summary['total_contracts']:,}")
    print(f"Average contract value: {summary['avg_contract_value']:.2f} model units")
    print(f"Composite indicator: {summary['final_efficiency']:.3f}")
    print(f"Change across recorded cycles: {summary['efficiency_improvement']:.3f}")
    print(f"Assignment rate: {summary['assignment_rate']:.3f}")
    print(f"Load-dispersion indicator: {summary['load_balance']:.3f}")
    print(f"Market activity indicator: {summary['market_efficiency']:.3f}")
    print(
        f"Active locations: {summary['active_locations']:,}/"
        f"{ecosystem.n_locations:,}"
    )
    print(f"Mean location reputation: {summary['avg_reputation']:.3f}")
    print("Detected pattern counts:")
    if summary["emergent_behaviors"]:
        for behavior_type, count in summary["emergent_behaviors"].items():
            print(f"  {behavior_type}: {count}")
    else:
        print("  No patterns were recorded.")


visualize_ecosystem_results(ecosystem)

## Analyze assignment patterns

The following analysis reports detected concentration events and examines where high-velocity products and selected seasonal categories were assigned. The percentages are descriptive summaries of the simulated assignments. Premium-location assignment is not a direct measure of distance, and the model does not move products after a contract is formed. Therefore, these measurements do not establish migration toward pick stations or prove that a particular pattern was caused by agent negotiation.


In [ ]:
def analyze_emergent_behaviors(ecosystem: AutonomousEcosystem) -> None:
    """Report recorded concentration events and assignment summaries."""
    if not ecosystem.emergent_behaviors:
        print("No concentration events were recorded during this run.")
    else:
        print("\nRecorded assignment-pattern events")
        print("=" * 60)
        for behavior_type in sorted(
            {behavior["type"] for behavior in ecosystem.emergent_behaviors}
        ):
            events = [
                behavior
                for behavior in ecosystem.emergent_behaviors
                if behavior["type"] == behavior_type
            ]
            print(f"\n{behavior_type}: {len(events)} event(s)")
            for event in events[:3]:
                print(f"  Time: {event['timestamp'].strftime('%H:%M:%S')}")
                print(f"  Description: {event['description']}")

    location_by_id = {
        location.agent_id: location for location in ecosystem.location_agents
    }
    high_velocity_products = [
        product
        for product in ecosystem.product_agents
        if product.is_high_velocity
    ]
    assigned_high_velocity = [
        product
        for product in high_velocity_products
        if product.current_location is not None
    ]
    premium_assigned = sum(
        location_by_id[product.current_location].location_type == "premium"
        for product in assigned_high_velocity
    )
    premium_share = (
        premium_assigned / len(assigned_high_velocity)
        if assigned_high_velocity
        else 0.0
    )

    print("\nHigh-velocity assignment summary")
    print(f"High-velocity products: {len(high_velocity_products):,}")
    print(f"Assigned high-velocity products: {len(assigned_high_velocity):,}")
    print(f"Assigned to premium locations: {premium_assigned:,}")
    print(f"Premium share among assigned high-velocity products: {premium_share:.1%}")

    seasonal_categories = {"seasonal", "sports", "luxury"}
    seasonal_assignments: Dict[str, Dict[str, int]] = defaultdict(
        lambda: defaultdict(int)
    )
    for product in ecosystem.product_agents:
        if (
            product.product_category in seasonal_categories
            and product.current_location is not None
        ):
            seasonal_assignments[product.current_location][
                product.product_category
            ] += 1

    concentrated_locations = 0
    for category_counts in seasonal_assignments.values():
        total = sum(category_counts.values())
        if total >= 3 and max(category_counts.values()) / total > 0.6:
            concentrated_locations += 1

    print("\nSelected seasonal-category concentration")
    print(f"Assigned locations with at least three selected products: "
          f"{sum(sum(counts.values()) >= 3 for counts in seasonal_assignments.values()):,}")
    print(
        "Locations meeting the within-group concentration rule: "
        f"{concentrated_locations:,}"
    )
    print(
        "These counts describe the assignment state; they do not measure "
        "accessibility or compare the market with a centralized optimizer."
    )


analyze_emergent_behaviors(ecosystem)

## Relationship to other SLAP approaches

This tier represents a different modeling perspective: assignment decisions arise from interactions among product agents, location agents, and an auction market rather than from a single centralized optimization calculation. The model uses bids, product-specific feasibility checks, utility scores, contracts, and reputation updates to represent those interactions.

The distinction is conceptual, not an empirical claim that this simulation outperforms mathematical programming, savings heuristics, metaheuristics, learned models, or digital-twin methods. Each approach has different assumptions, objectives, computational requirements, and validation needs. A fair performance comparison would require a shared instance, consistent objective and constraints, and measured results; those are outside this notebook's scope.

### Components represented

- **Location agents:** storage locations with capacity, speed, reliability, cost, specialization, and reputation.
- **Product agents:** products with category, velocity, acceptable cost, and minimum reliability.
- **Auction market:** collects sampled location offers and records winning prices.
- **Contracts:** represent accepted product-location assignments and consume one unit of location capacity.
- **Reputation updates:** move location reputation toward recent simulated success outcomes after enough observations.
- **Pattern indicators:** identify selected concentrations in the assignment state; they are descriptive and do not establish causation.

### Interpretation and possible extensions

Use the outputs to inspect how the specified local decision rules behave under the generated instance and demand scenario. They should not be interpreted as proof of centralized optimality, fault tolerance, linear scalability, a fixed percentage of optimal performance, or a measured adaptation-speed advantage. Natural extensions within this modeling approach include representing product size in capacity consumption, allowing contracts to expire and products to be reassigned, modeling order and travel costs in greater detail, and evaluating multiple generated instances. Each extension would require explicit assumptions and corresponding changes to the model and its measurements.
